# Population Exposure From Event BBoxes

This notebook:
- reads event metadata from `mas/beliefs/events.asl`
- matches each event to the appropriate WorldPop raster in `data/processed/exposure/`
- crops the raster to each event bbox
- computes population exposure metrics
- exports CSV and ASL facts for the reasoning layer

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import rasterio
from shapely.geometry import box
from pyproj import Geod


In [2]:
# Project paths
ROOT = Path("../..")
EVENTS_PATH = ROOT / 'mas' / 'beliefs' / 'events.asl'
POP_DIR = ROOT / 'data' / 'processed' / 'exposure'
CROPS_DIR = POP_DIR / 'cropped_to_events'
OUT_CSV = ROOT / 'data' / 'processed' / 'exposure' / 'population_exposure_summary.csv'
OUT_ASL = ROOT / 'mas' / 'beliefs' / 'population_exposure.asl'

CROPS_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
OUT_ASL.parent.mkdir(parents=True, exist_ok=True)

ROOT, EVENTS_PATH, POP_DIR

(PosixPath('../..'),
 PosixPath('../../mas/beliefs/events.asl'),
 PosixPath('../../data/processed/exposure'))

In [3]:
# Parse events, countries, and bboxes from events.asl
text = EVENTS_PATH.read_text(encoding='utf-8')

events = sorted(set(re.findall(r'^event\(([^)]+)\)\.$', text, flags=re.MULTILINE)))
country_map = {e: c for e, c in re.findall(r'^country\(([^,]+),\s*([^)]+)\)\.$', text, flags=re.MULTILINE)}
bbox_map = {
    e: tuple(map(float, (minx, miny, maxx, maxy)))
    for e, minx, miny, maxx, maxy in re.findall(
        r'^bbox\(([^,]+),\s*([\-0-9.]+),\s*([\-0-9.]+),\s*([\-0-9.]+),\s*([\-0-9.]+)\)\.$',
        text,
        flags=re.MULTILINE,
    )
}

parsed = []
for e in events:
    if e in country_map and e in bbox_map:
        parsed.append({
            'event': e,
            'country': country_map[e],
            'bbox': bbox_map[e],
        })

pd.DataFrame(parsed)

,event,country,bbox
0,ahr_valley,germany,"(6.950227, 50.471351, 7.18725, 50.566012)"
1,antalya,turkey,"(30.938848, 36.610847, 32.116923, 37.114228)"
2,emilia,italy,"(11.820598, 44.502971, 12.032391, 44.573716)"
3,evros,greece,"(25.144957, 40.781609, 26.258694, 41.221014)"
4,serra_de_estrela,portugal,"(-7.764595, 40.300913, -7.196839, 40.547715)"
5,sindh,pakistan,"(67.67564, 27.258053, 68.528303, 27.914313)"


In [4]:
# Build event -> cropped population raster mapping
# Expected naming like: ahr_valley_pop_crop.tif
raster_by_event = {}
for tif in sorted(CROPS_DIR.glob('*_pop_crop.tif')):
    event_id = tif.name.replace('_pop_crop.tif', '')
    raster_by_event[event_id] = tif

if not raster_by_event:
    print('No cropped event rasters were found in', CROPS_DIR)
    print('Expected files like ahr_valley_pop_crop.tif, emilia_pop_crop.tif, sindh_pop_crop.tif, etc.')

raster_by_event


{'ahr_valley': PosixPath('../../data/processed/exposure/cropped_to_events/ahr_valley_pop_crop.tif'),
 'antalya': PosixPath('../../data/processed/exposure/cropped_to_events/antalya_pop_crop.tif'),
 'emilia': PosixPath('../../data/processed/exposure/cropped_to_events/emilia_pop_crop.tif'),
 'evros': PosixPath('../../data/processed/exposure/cropped_to_events/evros_pop_crop.tif'),
 'serra_de_estrela': PosixPath('../../data/processed/exposure/cropped_to_events/serra_de_estrela_pop_crop.tif'),
 'sindh': PosixPath('../../data/processed/exposure/cropped_to_events/sindh_pop_crop.tif')}

In [5]:
# Compute population metrics from the existing per-event cropped rasters
geod = Geod(ellps='WGS84')
rows = []

for rec in parsed:
    event_id = rec['event']
    country = rec['country']
    minx, miny, maxx, maxy = rec['bbox']

    crop_path = raster_by_event.get(event_id)
    if crop_path is None:
        print(f'Skipping {event_id}: no cropped raster found in {CROPS_DIR}')
        continue

    geom_wgs84 = box(minx, miny, maxx, maxy)

    with rasterio.open(crop_path) as src:
        arr = src.read(1).astype(float)
        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan

        pop_total = float(np.nansum(arr))
        valid_pixels = int(np.sum(~np.isnan(arr)))

    area_m2, _ = geod.geometry_area_perimeter(geom_wgs84)
    area_km2 = abs(area_m2) / 1e6
    density = pop_total / area_km2 if area_km2 > 0 else np.nan

    rows.append({
        'event': event_id,
        'country': country,
        'population_raster': crop_path.name,
        'total_population': round(pop_total, 2),
        'area_km2': round(area_km2, 4),
        'population_density_km2': round(float(density), 2),
        'valid_pixels': valid_pixels,
        'crop_path': str(crop_path.relative_to(ROOT)).replace('\\', '/'),
    })

if rows:
    df = pd.DataFrame(rows).sort_values('event').reset_index(drop=True)
else:
    df = pd.DataFrame(columns=[
        'event',
        'country',
        'population_raster',
        'total_population',
        'area_km2',
        'population_density_km2',
        'valid_pixels',
        'crop_path',
    ])
    print('No population exposure rows were created because no matching cropped rasters were available.')
df


,event,country,population_raster,total_population,area_km2,population_density_km2,valid_pixels,crop_path
0,ahr_valley,germany,ahr_valley_pop_crop.tif,38365.01,177.0092,216.74,9159,data/processed/exposure/cropped_to_events/ahr_...
1,antalya,turkey,antalya_pop_crop.tif,465543.15,5868.3958,79.33,106724,data/processed/exposure/cropped_to_events/anta...
2,emilia,italy,emilia_pop_crop.tif,15909.53,132.3286,120.23,11262,data/processed/exposure/cropped_to_events/emil...
3,evros,greece,evros_pop_crop.tif,167135.48,4572.4145,36.55,117438,data/processed/exposure/cropped_to_events/evro...
4,serra_de_estrela,portugal,serra_de_estrela_pop_crop.tif,69997.87,1320.4370,53.01,68839,data/processed/exposure/cropped_to_events/serr...
5,sindh,pakistan,sindh_pop_crop.tif,3828128.98,6122.3091,625.28,260953,data/processed/exposure/cropped_to_events/sind...


In [6]:
# Exposure classification for reasoning
# You can adjust thresholds later based on project needs.
def exposure_class(d):
    if np.isnan(d):
        return 'unknown'
    if d < 100:
        return 'low'
    if d < 500:
        return 'medium'
    return 'high'

df['population_exposure_class'] = df['population_density_km2'].apply(exposure_class)
df

,event,country,population_raster,total_population,area_km2,population_density_km2,valid_pixels,crop_path,population_exposure_class
0,ahr_valley,germany,ahr_valley_pop_crop.tif,38365.01,177.0092,216.74,9159,data/processed/exposure/cropped_to_events/ahr_...,medium
1,antalya,turkey,antalya_pop_crop.tif,465543.15,5868.3958,79.33,106724,data/processed/exposure/cropped_to_events/anta...,low
2,emilia,italy,emilia_pop_crop.tif,15909.53,132.3286,120.23,11262,data/processed/exposure/cropped_to_events/emil...,medium
3,evros,greece,evros_pop_crop.tif,167135.48,4572.4145,36.55,117438,data/processed/exposure/cropped_to_events/evro...,low
4,serra_de_estrela,portugal,serra_de_estrela_pop_crop.tif,69997.87,1320.4370,53.01,68839,data/processed/exposure/cropped_to_events/serr...,low
5,sindh,pakistan,sindh_pop_crop.tif,3828128.98,6122.3091,625.28,260953,data/processed/exposure/cropped_to_events/sind...,high


In [7]:
# Export CSV summary
df.to_csv(OUT_CSV, index=False)
OUT_CSV

PosixPath('../../data/processed/exposure/population_exposure_summary.csv')

In [8]:
# Export ASL facts for MAS reasoning
lines = []
lines.append('// ----------------------------------------------')
lines.append('// Population exposure indicators')
lines.append('// Auto-generated from eo_processing/population/population_exposure_from_bbox.ipynb')
lines.append('// ----------------------------------------------\n')

if df.empty:
    print('Skipping ASL export because the population exposure table is empty.')
else:
    for _, r in df.iterrows():
        e = r['event']
        lines.append(f"total_population({e}, {r['total_population']:.2f}).")
        lines.append(f"population_area_km2({e}, {r['area_km2']:.4f}).")
        lines.append(f"population_density_km2({e}, {r['population_density_km2']:.2f}).")
        lines.append(f"population_exposure_class({e}, {r['population_exposure_class']}).")
        lines.append(f"population_valid_pixels({e}, {int(r['valid_pixels'])}).")
        lines.append('')

    OUT_ASL.write_text('\n'.join(lines), encoding='utf-8')
OUT_ASL

PosixPath('../../mas/beliefs/population_exposure.asl')